In [1]:
# ===== Célula 1 — Amostra de 20 avaliações com o macrotema atribuído =====
import json
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
catalogo = json.loads((RAIZ / "resultados" / "temas.json").read_text(encoding="utf-8"))
macros = {m["id"]: m for m in catalogo["macrotemas"]}
temas = pd.read_parquet(RAIZ / "data" / "processed" / "temas.parquet")
corpus = pd.read_parquet(RAIZ / "data" / "processed" / "corpus.parquet",
                         columns=["review_id", "texto"])
dados = temas.merge(corpus, on="review_id")

# 10 avaliações sorteadas de cada grupo (problema e elogio)
amostra = pd.concat([g.sample(10, random_state=42) for _, g in dados.groupby("grupo")],
                    ignore_index=True)
for i, r in amostra.iterrows():
    print(f"#{i + 1:02d} [{r['grupo']}] nota {r['nota']} | MACROTEMA: {macros[r['macrotema']]['nome']}")
    print(f'     "{r["texto"]}"\n')

#01 [elogio] nota 4 | MACROTEMA: Avaliações mistas ou genéricas
     "Obrigado"

#02 [elogio] nota 5 | MACROTEMA: Elogio à entrega rápida
     "Super recomendo. Chegou certinho e bem embalado, e teve um bom preço, compraria novamente na loja."

#03 [elogio] nota 5 | MACROTEMA: Qualidade e atendimento geral
     "Ótimo. Muito Bom, Produto, entrega"

#04 [elogio] nota 5 | MACROTEMA: Qualidade dos produtos
     "Ótimo. -"

#05 [elogio] nota 5 | MACROTEMA: Elogio à entrega rápida
     "Como sempre as lojas lannister surpreende tanto no atendimento quanto no prazo de entrega, entregou meu produto antes da data prevista. Super recomendo!"

#06 [elogio] nota 5 | MACROTEMA: Avaliações mistas ou genéricas
     "Recebi o produto dentro do prazo,veio embalado e não tinha danificados. Porem, a loja targaryen não atualizava o status do pedido nem no App das lojas lannister e nem no site dos correios."

#07 [elogio] nota 5 | MACROTEMA: Elogio à entrega rápida
     "Chegou perfeita, super recomendo"


In [5]:
# ===== Célula 2 — Seus vereditos (S = macrotema certo, N = errado) =====
vereditos = ["S", "N", "S", "N", "S", "S", "N", "S", "S", "N",
             "S", "S", "S", "S", "S", "N", "S", "S", "S", "S"]
assert len(vereditos) == len(amostra), "um veredito para cada avaliação"
assert all(v in ("S", "N") for v in vereditos), "use apenas S ou N"

amostra["correto"] = [v == "S" for v in vereditos]
print(f"Macrotema correto: {amostra['correto'].mean():.0%} ({amostra['correto'].sum()} de 20)")
print(amostra.groupby("grupo")["correto"].mean().map("{:.0%}".format).to_string())
print("\nErradas:")
for i, r in amostra[~amostra["correto"]].iterrows():
    print(f"  #{i + 1:02d} [{r['grupo']}] {macros[r['macrotema']]['nome']} | {r['texto'][:90]}")
amostra.to_csv(RAIZ / "resultados" / "validacao_temas.csv", index=False, encoding="utf-8")
print("\nValidação salva em resultados/validacao_temas.csv")

Macrotema correto: 75% (15 de 20)
grupo
elogio      60%
problema    90%

Erradas:
  #02 [elogio] Elogio à entrega rápida | Super recomendo. Chegou certinho e bem embalado, e teve um bom preço, compraria novamente 
  #04 [elogio] Qualidade dos produtos | Ótimo. -
  #07 [elogio] Elogio à entrega rápida | Chegou perfeita, super recomendo
  #10 [elogio] Qualidade dos produtos | Excelente
  #16 [problema] Problemas de logística e frete | O produto foi entregue, porém, com o prazo de validade para vencimento em dois dias após a

Validação salva em resultados/validacao_temas.csv
